RAG pipeline
Phase 0 — Project framing (1–2 hours)

Goal: make the project coherent, not random.

Work items

 Decide use case (1 sentence)

e.g. “Chat with uploaded PDFs using a local LLM (RAG)”

 Decide input type

PDF only (recommended)

 Decide scope boundary

Single user

Local storage

No auth

📌 Output:

Project title

One-paragraph description (you’ll reuse this everywhere)

Phase 1 — Basic RAG pipeline (core logic) (1–2 days)

Goal: make retrieval + generation work before any UI.

1. Document ingestion

Work items

 Load PDF (e.g. PyPDFLoader)

 Clean text (remove headers/footers if needed)

 Chunk text

fixed size (e.g. 500–800 tokens)

overlap (e.g. 100)

2. Embeddings + vector store

Work items

 Load embedding model (MiniLM)

 Generate embeddings for chunks

 Store in FAISS / Chroma

 Save index locally

3. Retriever

Work items

 Implement similarity search

 Tune top_k (e.g. 3–5)

 Return text + metadata (source page)

4. LLM integration

Work items

 Connect to local/free LLM

 Prompt template:

system instruction

retrieved context

user question

 Generate answer

📌 Output:

Python script where you can do:

ask("What is X?")


If this works → you’re 60% done.

Phase 2 — Backend API (FastAPI) (1 day)

Goal: expose RAG as a service.

API endpoints

Work items

 POST /upload

Upload PDF

Trigger ingestion + embedding

 POST /chat

Input: user question

Output: answer + sources

Backend structure

Work items

 Separate folders:

rag/ (logic)

api/ (routes)

 Error handling

No docs uploaded

Empty retrieval

📌 Output:

You can call RAG via HTTP

Phase 3 — Simple chatbot UI (0.5–1 day)

Goal: demonstrate usability, not design skills.

UI features

Work items

 File upload

 Chat input box

 Chat history display

 “Thinking…” state

Recommended

Streamlit (fastest)

or minimal React page

📌 Output:

User uploads PDF

User chats with it

Phase 4 — Quality improvements (choose 1–2 only) (0.5–1 day)

Goal: elevate from “toy” to “interview-ready”.

Pick one or two:

 Show retrieved sources under each answer

 Add prompt guardrails (“Answer only from context”)

 Adjustable top_k

 Simple logging (question → retrieved chunks)

Phase 5 — Documentation (VERY important) (0.5 day)

Goal: make recruiters understand your thinking.

README sections

Work items

 Problem statement

 Architecture diagram

 RAG pipeline explanation

 Design decisions

 Limitations

 Future improvements

📌 This is where you flex understanding, not code.

Phase 6 — Polish & publish (0.5 day)

Work items

 Clean repo structure

 requirements.txt

 Example PDF

 Screenshots / short GIF

 Clear setup instructions

Final deliverables (what you’ll have)

✅ Working RAG chatbot

✅ Local/free models (no API dependency)

✅ Clean system design

✅ Interview-ready explanations

✅ Strong GitHub project

#API 
Upload file
RAG Pipeline
Send Message
Reply Message
Message Log
CRUD files per prompt

In [ ]:
#Load pdf

#Data preprocessing

#Chunking

#Embeddings

#Store in vector db

#Retrieval

#Prompt templates

#Model generation

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json
import faiss
pdf_path = "snow-white.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      
    chunk_overlap=100,  
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)



model = SentenceTransformer('all-MiniLM-L6-v2')


texts = [chunk.page_content for chunk in chunks]


embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  
index.add(embeddings)

print("FAISS index size:", index.ntotal)


# Save FAISS index
faiss.write_index(index, "faiss_index.bin")
print("FAISS index saved to faiss_index.bin")

# Save chunk metadata
chunks_data = [
    {"text": chunk.page_content, "metadata": getattr(chunk, "metadata", {})}
    for chunk in chunks
]

with open("chunks_metadata.json", "w", encoding="utf-8") as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)
print("Chunk metadata saved to chunks_metadata.json")
# Load FAISS index
index = faiss.read_index("faiss_index.bin")
print("FAISS index loaded")

# Load chunk metadata
with open("chunks_metadata.json", "r", encoding="utf-8") as f:
    chunks_data = json.load(f)
print("Chunk metadata loaded")

c:\Users\cyoff\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 2/2 [00:00<00:00,  2.57it/s]

Embeddings shape: (61, 384)
FAISS index size: 61
FAISS index saved to faiss_index.bin
Chunk metadata saved to chunks_metadata.json
FAISS index loaded
Chunk metadata loaded


In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
import uuid

# -----------------------------
# 1️⃣ Load PDF
# -----------------------------
pdf_path = "snow-white.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# -----------------------------
# 2️⃣ Split text
# -----------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

texts = [chunk.page_content for chunk in chunks]

# -----------------------------
# 3️⃣ Prepare IDs and metadata
# -----------------------------
ids = [str(uuid.uuid4()) for _ in texts]

metadatas = [
    {
        "source": pdf_path,
        "page": chunk.metadata.get("page", None)
    }
    for chunk in chunks
]

# -----------------------------
# 4️⃣ Initialize Chroma (safe mode)
# -----------------------------
client = chromadb.Client(
    Settings(
        persist_directory="./chroma_db",
        anonymized_telemetry=False
    )
)

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="documents",
    embedding_function=embedding_function
)

# -----------------------------
# 5️⃣ Store in Chroma
# -----------------------------
collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas
)

print(f"✅ Stored {len(texts)} chunks in ChromaDB")


c:\Users\cyoff\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [7]:
import requests
from sentence_transformers import SentenceTransformer
import faiss
import json

# --- 1️⃣ Load FAISS index and chunk metadata ---
index = faiss.read_index("faiss_index.bin")
with open("chunks_metadata.json", "r", encoding="utf-8") as f:
    chunks_data = json.load(f)

# --- 2️⃣ Load embedding model ---
model = SentenceTransformer("all-MiniLM-L6-v2")

# --- 3️⃣ Define the question ---
question = "Why the stepmother wants to kill Snowwhite?"

# --- 4️⃣ Embed the question ---
query_vec = model.encode([question], convert_to_numpy=True, normalize_embeddings=True)

# --- 5️⃣ Retrieve top-k similar chunks ---
top_k = 4
D, I = index.search(query_vec, top_k)
retrieved_chunks = [chunks_data[i] for i in I[0]]

# --- 6️⃣ Build RAG prompt ---
context = "\n\n".join([c["text"] for c in retrieved_chunks])
prompt = f"""
You are a helpful assistant.
Answer ONLY using the context below.
If the answer is not contained in the context, say "I don't know."

Context:
{context}

Question:
{question}
"""

# --- 7️⃣ Call Ollama API ---
url = "http://localhost:11434/v1/chat/completions"  # your Ollama API endpoint
model_name = "qwen3-coder:480b-cloud"  # your model

payload = {
    "model": model_name,
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.1
}

response = requests.post(url, json=payload)
response.raise_for_status()

# --- 8️⃣ Print the answer ---
answer = response.json()["choices"][0]["message"]["content"]
print("Answer:", answer)


Answer: The stepmother wants to kill Snow-White because she is envious of Snow-White's beauty. The Queen's envy gives her no peace, day or night, and hatred for the child fills her heart. She cannot bear the sight of Snow-White and vows that Snow-White shall die, even if it costs her own life.
